<a href="https://colab.research.google.com/github/Clueles2/A-Search-Algorithm/blob/main/Final_A_Search_Route_Finding_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Node Map



In [ ]:
import math
import heapq

node_map = {
    "Research": {"Storage": 14},
    "Storage": {"Research": 14, "Containment": 58, "Emergency": 60},
    "Containment": {"Storage": 58, "Medical": 20, "Control": 100, "Airlock": 68},
    "Medical": {"Containment": 20, "MainLab": 96},
    "MainLab": {"Medical": 96, "Airlock": 15},
    "Airlock": {"Development": 12, "Observation": 10},
    "Emergency": {"Storage": 60, "Development": 71},
    "Control": {"Containment": 100, "Chemical": 11},
    "Development": {"Security": 35, "Emergency": 71, "BioGarden": 10},
    "BioGarden": {},
    "Security": {"Development": 35, "Hub": 18, "Observation": 44},
    "Hub": {"Security": 18, "Workshop": 45},
    "Workshop": {"Hub": 45, "Electrical": 19},
    "Electrical": {"Workshop": 19, "Restriction": 71},
    "Observation": {"Security": 44, "Restriction": 64, "Chemical": 41},
    "Chemical": {"Observation": 41, "Control": 11},
    "Restriction": {"Observation": 64, "Electrical": 71, "DataArchive": 14},
    "DataArchive": {"Restriction": 14}
}
#Airlock can only go to Development or Observation
#BioGarden is a dead end

node_position = {
    "Research": (-18.0, 16.0),
    "Storage": (-11.0, 16.9),
    "Containment": (4.0, 25.0),
    "Medical": (1.5, 12.7),
    "MainLab": (13.0, 9.0),
    "Airlock": (20.0, 18.7),
    "Emergency": (0.0, 0.0),
    "Control": (-3.0, 44.0),
    "Development": (23.0, 0.0),
    "Security": (28.0, 25.0),
    "Hub": (35.0, 28.0),
    "Workshop": (42.0, 15.0),
    "Electrical": (53.0, 29.0),
    "Observation": (26.0, 34.8),
    "Chemical": (13.2, 41.0),
    "Restriction": (40.2, 34.8),
    "DataArchive": (53.0, 44.0),
    "BioGarden": (56.0, 45.0)
}


#Node definition


In [ ]:
class Node:

  def __init__(self, room_name, x, y):
    self.room_name = room_name
    self.neighbours = []
    self.parent = None
    self.parent_cost = 0
    self.x = x
    self.y = y

    self.g = 0
    self.f = 0

  def get_room_name(self):
    return self.room_name

  def get_neighbours(self):
    return self.neighbours

  def set_neighbour(self, neighbour, cost):
    self.neighbours.append((neighbour, cost))

  def set_parent(self, parent, parent_cost):
    self.parent = parent
    self.parent_cost = parent_cost

  def get_parent(self):
    return self.parent, self.parent_cost

  def get_f(self):
    return self.f

  def set_f(self, f):
    self.f = f

  def get_g(self):
    return self.g

  def set_g(self, g):
    self.g = g

  def get_x(self):
    return self.x

  def get_y(self):
    return self.y

  def get_position(self):
    return (self.x, self.y)

  def __repr__(self):
    return f"({self.x}, {self.y}) , {self.room_name}"

  def __eq__(self, other):
    return self.room_name == other.room_name

  def __hash__(self):
    return hash(self.room_name)


#Heuristic function

In [ ]:
def h(node1=Node,node2=Node):
  dx = node1.get_x() - node2.get_x()
  dy = node1.get_y() - node2.get_y()
  return 2 * (abs(dx) + abs(dy))

Takes two nodes and calculates the difference in their x and y coordinates.
Uses the maps scale, 1% oxygen depletion per 0.5 unit, to estimate the depletion over this distance.

#A* Search

In [ ]:
def search(nodes, start_node, end_node, limit):

  #priority queue containing the nodes to next consider: [f, room_name, node]
  open_list = []

  #set of nodes already fully explored
  explored_set = set()

  #tracks the oxygen costs from start_node to node: {node: cost to get here}
  g_costs = {}

  #adds start_node to priority queue and initialises f and g to 0
  heapq.heappush(open_list, (0, start_node.get_room_name(), start_node))
  g_costs[start_node] = 0

  while open_list:
    f, name, current_node = heapq.heappop(open_list)

    if current_node in explored_set:
      print("Already explored", current_node.get_room_name())
      continue

    explored_set.add(current_node)

    print("Current node: ", current_node.get_room_name(), "f: ",
          current_node.get_f(), "g: ", current_node.get_g())

    #checks if end_node has been reached
    if current_node == end_node:

      path = []
      total_cost = 0

      #creates a path by appending the end_node(current_node) to the path, then
      #following the parent links(nodes that came before) to the start_node.
      while current_node != start_node:
        path.append(current_node.get_room_name())
        current_node, parent_cost = current_node.get_parent()
        total_cost += parent_cost

      path.append(start_node.get_room_name())
      return path[::-1], total_cost

    #explores each neighbour of the current_node
    for neighbour, cost in current_node.get_neighbours():

      #computes cost from start_node to neighbour node (current g + cost)
      neighbour_g_estimate = g_costs[current_node] + cost
      neighbour_h_estimate = h(neighbour, end_node)
      neighbour_f_estimate = neighbour_g_estimate + neighbour_h_estimate

      # if the neighbour exists in the g_cost and the recently calculated g cost
      # is greater - discard this option. if it exceeds oxygen limit, also discard.
      if neighbour in g_costs:
        if neighbour_g_estimate >= g_costs[neighbour] or neighbour_g_estimate >= limit:
          continue

      if neighbour in explored_set:
        continue

      #if the f value exceeds limit, then discard since there is no point further
      #exploring a path that is estimated to exceed the oxygen limit.
      if neighbour_f_estimate >= limit:
        print("Exceeding limit")
        continue

      #the node's f and g are updated and parent node(current_node) is set
      g_costs[neighbour] = neighbour_g_estimate
      neighbour.set_parent(current_node, cost)
      neighbour.set_f(neighbour_f_estimate)
      neighbour.set_g(neighbour_g_estimate)

      #prints 'qualified' neighbour nodes that don't exceed limit/already
      #explored or have the current best g value
      print("Neighbour:", neighbour.get_room_name(), "f: ", neighbour.get_f(),
            "g: ", neighbour.get_g())

      heapq.heappush(open_list, (
          neighbour_f_estimate, neighbour.get_room_name(), neighbour))

  return None, None

#Main run function

In [ ]:
def run(start_node, end_node, limit):
  nodes = {}

  #creates node objects from each location in node_map
  for name in node_map:
    x, y = node_position[name]
    nodes[name] = Node(room_name=name, x=x, y=y)
    #In the format {room_name : Node object}

  for name, neighbours in node_map.items():
    node = nodes[name]

    #looking at the neighbours e.g. {"Medical": 96, "Airlock": 5}, adds each
    # neighbour (neighbour, cost) to the neighbour list of each node
    for neighbour_name, cost in neighbours.items():
      node.set_neighbour(nodes[neighbour_name], cost)


  start_node = nodes[start_node]
  end_node = nodes[end_node]

  print("START NODE:", start_node)
  print("END NODE:", end_node)
  print()

  path, total_cost = search(nodes, start_node, end_node, limit)
  print()
  print("Optimal path:", path)
  print("Total cost:", total_cost)


Parses the node_map into nodes, sets the neighbours and their costs of each node, and runs the search algorithm. run() takes the parameters: start_node, end_node and a limit. The limit parameter is also passed into search() for ease of testing.

#Test 1: Standard inputs




In [ ]:
run("Research", "DataArchive", 500)


START NODE: (-18.0, 16.0) , Research
END NODE: (53.0, 44.0) , DataArchive

Current node:  Research f:  0 g:  0
Neighbour: Storage f:  196.2 g:  14
Current node:  Storage f:  196.2 g:  14
Neighbour: Containment f:  208.0 g:  72
Neighbour: Emergency f:  268.0 g:  74
Current node:  Containment f:  208.0 g:  72
Neighbour: Medical f:  257.6 g:  92
Neighbour: Control f:  284.0 g:  172
Neighbour: Airlock f:  256.6 g:  140
Current node:  Airlock f:  256.6 g:  140
Neighbour: Development f:  300.0 g:  152
Neighbour: Observation f:  222.4 g:  150
Current node:  Observation f:  222.4 g:  150
Neighbour: Security f:  282.0 g:  194
Neighbour: Restriction f:  258.0 g:  214
Neighbour: Chemical f:  276.6 g:  191
Current node:  Medical f:  257.6 g:  92
Neighbour: MainLab f:  338.0 g:  188
Current node:  Restriction f:  258.0 g:  214
Neighbour: Electrical f:  315.0 g:  285
Neighbour: DataArchive f:  228.0 g:  228
Current node:  DataArchive f:  228.0 g:  228

Optimal path: ['Research', 'Storage', 'Containm

Here, the algorithm finds the optimal path from Research to Data Archives. At the Observation room, despite having two neighbours that cost less(Security, Chemical), the algorithm chooses the neighbour that costs the most (Restriction), because it has the lowest f estimate. However, before exploring the Restriction route,it must first consider the Medical route as it was the next cheapest option.

#Test 2: One-way road

In [ ]:
run("Observation", "MainLab", 500)

START NODE: (26.0, 34.8) , Observation
END NODE: (13.0, 9.0) , MainLab

Current node:  Observation f:  0 g:  0
Neighbour: Security f:  106.0 g:  44
Neighbour: Restriction f:  170.0 g:  64
Neighbour: Chemical f:  105.4 g:  41
Current node:  Chemical f:  105.4 g:  41
Neighbour: Control f:  154.0 g:  52
Current node:  Security f:  106.0 g:  44
Neighbour: Development f:  117.0 g:  79
Neighbour: Hub f:  144.0 g:  62
Current node:  Development f:  117.0 g:  79
Neighbour: Emergency f:  194.0 g:  150
Neighbour: BioGarden f:  247.0 g:  89
Current node:  Hub f:  144.0 g:  62
Neighbour: Workshop f:  177.0 g:  107
Current node:  Control f:  154.0 g:  52
Neighbour: Containment f:  202.0 g:  152
Current node:  Restriction f:  170.0 g:  64
Neighbour: Electrical f:  255.0 g:  135
Neighbour: DataArchive f:  228.0 g:  78
Current node:  Workshop f:  177.0 g:  107
Neighbour: Electrical f:  246.0 g:  126
Current node:  Emergency f:  194.0 g:  150
Neighbour: Storage f:  273.8 g:  210
Current node:  Containm

Ideally, the shortest path from from Observation to Main Lab passes through Airlock and costs: 10 + 15 = 25. However, Airlock is not accessible from Observation so the algorithm must choose the longer route - passing through chemical instead.

#Test 3: Dead end


In [ ]:
run("BioGarden", "Development", 500)

START NODE: (56.0, 45.0) , BioGarden
END NODE: (23.0, 0.0) , Development

Current node:  BioGarden f:  0 g:  0

Optimal path: None
Total cost: None


Returns no path as Bio Garden is a dead end.

#Test 4: Exceeding limit

In [ ]:
run("DataArchive", "MainLab", 300)
print()
run("DataArchive", "MainLab", 500)

START NODE: (53.0, 44.0) , DataArchive
END NODE: (13.0, 9.0) , MainLab

Current node:  DataArchive f:  0 g:  0
Neighbour: Restriction f:  120.0 g:  14
Current node:  Restriction f:  120.0 g:  14
Neighbour: Observation f:  155.6 g:  78
Neighbour: Electrical f:  205.0 g:  85
Current node:  Observation f:  155.6 g:  78
Neighbour: Security f:  184.0 g:  122
Neighbour: Chemical f:  183.4 g:  119
Current node:  Chemical f:  183.4 g:  119
Neighbour: Control f:  232.0 g:  130
Current node:  Security f:  184.0 g:  122
Neighbour: Development f:  195.0 g:  157
Neighbour: Hub f:  222.0 g:  140
Current node:  Development f:  195.0 g:  157
Neighbour: Emergency f:  272.0 g:  228
Exceeding limit
Current node:  Electrical f:  205.0 g:  85
Neighbour: Workshop f:  174.0 g:  104
Current node:  Workshop f:  174.0 g:  104
Current node:  Hub f:  222.0 g:  140
Current node:  Control f:  232.0 g:  130
Neighbour: Containment f:  280.0 g:  230
Current node:  Emergency f:  272.0 g:  228
Exceeding limit
Current no

This is run with 2 limit values, 300 and 500. With a higher limit, the algorithm returns a path. However, lowering the limit returns none as the algorithm no longer considers nodes that are estimated (f value) to exceed or have already exceeded 300. This is shown when the current node is Development, only the Emergency node is listed as a qualified neighbour because the other node's f value is an 'Exceeding limit'. Compared to the search with a 500 limit, when the current node is Development, Bio Garden is shown as a neighbour node.

#Test 5: Edge case

In [ ]:
run("Research", "Research", 500)

START NODE: (-18.0, 16.0) , Research
END NODE: (-18.0, 16.0) , Research

Current node:  Research f:  0 g:  0

Optimal path: ['Research']
Total cost: 0


Total cost is 0 as this is the start node.